# KWRI Spatial XGBoost Landslide Susceptibility Workflow
**Automated End-to-End Machine Learning Pipeline for Landslide Susceptibility Mapping**

This notebook implements a leak-free spatial machine learning pipeline using **XGBoost** to model landslide susceptibility:
- **Spatial Autocorrelation Control**: Uses spatial K-Means clustering and `GroupKFold` / `GroupShuffleSplit` to prevent training/testing contamination across geographic space.
- **Leakage-Safe Preprocessing**: Imputes numeric predictors using fold-specific medians, handles infinite values, and one-hot encodes NLCD land cover categories within an integrated `scikit-learn` Pipeline.
- **Class Imbalance Handling**: Automatically calculates and applies `scale_pos_weight` to the XGBoost booster.
- **Multi-Level Evaluation**: Computes pooled out-of-fold spatial CV metrics, an independent 30% spatial holdout evaluation, detailed confusion matrices with specificity/recall, and ROC-AUC curves.
- **Explainability & Artifacts**: Produces detailed and grouped feature importances, saves trained `.joblib` pipelines, and generates publication-grade visualizations (300 DPI).

---


### Workflow Overview & Environment Setup

Import essential libraries for numerical computing, spatial clustering, pipeline modeling, evaluation metrics, and visualization.
OpenMP threads are set to 4 to optimize XGBoost multithreading and prevent CPU oversubscription.


In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"

import json
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    cross_validate,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
)
from xgboost import XGBClassifier


### 1. Paths and Model Settings

Configure reproducible random seeds, spatial partitioning parameters (10 clusters, 5 folds, 30% spatial holdout), and input/output directory paths.

> **Note**: This cell automatically checks the local workspace folder first for `KWRI_latest_points_export.csv`, with fallback to the configured path.


In [ ]:
SEED = 42
NUMBER_OF_SPATIAL_GROUPS = 10
NUMBER_OF_CV_FOLDS = 5
SPATIAL_HOLDOUT_GROUP_FRACTION = 0.30
CLASSIFICATION_THRESHOLD = 0.50
MISSING_LANDCOVER_CODE = -9999

# Configure project base path
# Checks current working directory first, with fallback to original project path
default_project_base = Path(r'C:\Users\Utkarsha.Basnet\Documents\ChatGPT\KWRI')
workspace_base = Path.cwd()

# Locate input dataset
default_input = default_project_base / 'KWRI_Materials' / '02_Modeling_Data' / 'XGBoost_Modeling' / 'KWRI_latest_points_export.csv'
workspace_input = workspace_base / 'KWRI_latest_points_export.csv'

if default_input.exists():
    base = default_project_base
    input_csv = default_input
    output_dir = base / 'KWRI_Materials' / '05_Results_and_Maps' / 'XGBoost_KWRI_Spatial_Final'
elif workspace_input.exists():
    base = workspace_base
    input_csv = workspace_input
    output_dir = workspace_base / 'outputs' / 'XGBoost_KWRI_Spatial_Final'
else:
    # Fallback to local workspace paths (user can drop CSV here or customize path below)
    base = workspace_base
    input_csv = workspace_input
    output_dir = workspace_base / 'outputs' / 'XGBoost_KWRI_Spatial_Final'

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print('Input dataset path:', input_csv)
print('Output folder path:', output_dir)

if not input_csv.exists():
    print(f'\n[!] NOTE: Input CSV was not found at:\n    {input_csv}\n'
          f'Please place \'KWRI_latest_points_export.csv\' in the notebook directory or modify \'input_csv\'.')


### 2. Load and Validate Data

Load the points export CSV, strip whitespace from columns, validate presence of the target (`Labels`), spatial coordinates (`POINT_X`, `POINT_Y`), and topographic/environmental predictors, and verify binary class labels {0, 1}.


In [ ]:
df = pd.read_csv(input_csv)

df.columns = [
    str(column).strip()
    for column in df.columns
]

target_field = "Labels"

coordinate_fields = [
    "POINT_X",
    "POINT_Y",
]

numeric_features = [
    "Curvature",
    "TWI",
    "Dist_Road_KYTC_1",
    "Plan_Curv",
    "Profile_Curv",
    "Distan_to_Rivers",
    "Slope_KWRI",
    "Aspect_KWRI",
    "Rain_Mean_KWRI",
    "NDVI_Mean_KWRI",
]

categorical_features = [
    "Landcover_KWRI",
]

raw_features = (
    numeric_features
    + categorical_features
)

required_fields = (
    [target_field]
    + coordinate_fields
    + raw_features
)

missing_fields = [
    field
    for field in required_fields
    if field not in df.columns
]

if missing_fields:
    raise ValueError(
        "Required fields are missing:\n"
        + "\n".join(missing_fields)
    )

y = pd.to_numeric(
    df[target_field],
    errors="raise"
).astype(int)

unexpected_labels = (
    set(y.unique()) - {0, 1}
)

if unexpected_labels:
    raise ValueError(
        f"Unexpected label values: {unexpected_labels}"
    )

X_raw = df[raw_features].copy()

for field in numeric_features:
    X_raw[field] = pd.to_numeric(
        X_raw[field],
        errors="coerce"
    )

X_raw["Landcover_KWRI"] = pd.to_numeric(
    X_raw["Landcover_KWRI"],
    errors="coerce"
)

coordinates = (
    df[coordinate_fields]
    .apply(pd.to_numeric, errors="coerce")
)

if coordinates.isna().any().any():
    raise ValueError(
        "POINT_X or POINT_Y contains missing values."
    )

print("\nNumber of records:")
print(len(df))

print("\nClass distribution:")
print(y.value_counts().sort_index())


### 3. Predictor Data-Quality Report

Scan all numeric and categorical features for infinite values (±inf), NaN values, and finite counts, exporting a comprehensive data quality audit to CSV.


In [ ]:
quality_rows = []

for feature in raw_features:
    values = pd.to_numeric(
        X_raw[feature],
        errors="coerce"
    ).to_numpy(dtype=float)

    quality_rows.append({
        "feature": feature,
        "missing_count":
            int(np.isnan(values).sum()),
        "positive_infinity_count":
            int(np.isposinf(values).sum()),
        "negative_infinity_count":
            int(np.isneginf(values).sum()),
        "finite_count":
            int(np.isfinite(values).sum()),
    })

data_quality_report = pd.DataFrame(
    quality_rows
)

data_quality_report.to_csv(
    output_dir
    / "xgboost_predictor_data_quality.csv",
    index=False
)

print("\nPredictor data quality:")
display(data_quality_report)


### 4. Class Distribution

Calculate and visualize the ratio of non-landslide (0) to landslide (1) inventory points to inspect class imbalance.


In [ ]:
class_distribution = (
    y.value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="point_count")
)

class_distribution["class_name"] = (
    class_distribution["label"].map({
        0: "Non-Landslide",
        1: "Landslide",
    })
)

class_distribution.to_csv(
    output_dir
    / "xgboost_class_distribution.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(6, 5))

bars = ax.bar(
    class_distribution["class_name"],
    class_distribution["point_count"],
    color=["#4C78A8", "#C44E52"]
)

ax.set_ylabel("Number of Points")

ax.set_title(
    "Landslide Inventory Class Distribution",
    fontweight="bold"
)

ax.bar_label(
    bars,
    padding=4
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.30
)

ax.set_axisbelow(True)
fig.tight_layout()

fig.savefig(
    output_dir
    / "xgboost_class_distribution.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)


### 5. Class-Balance Weight Calculation

Compute the inverse class ratio `scale_pos_weight = negative_count / positive_count` to feed directly into XGBoost's loss formulation, balancing minority landslide penalties.


In [ ]:
negative_count = int((y == 0).sum())
positive_count = int((y == 1).sum())

if positive_count == 0:
    raise ValueError(
        "No landslide records were found."
    )

scale_pos_weight = (
    negative_count / positive_count
)

print("\nNon-landslide points:", negative_count)
print("Landslide points:", positive_count)

print(
    "scale_pos_weight:",
    scale_pos_weight
)


### 6. Leakage-Safe Model Pipeline

Construct a scikit-learn `Pipeline` encapsulating:
- **Numeric branch**: infinite-to-NaN conversion followed by training-fold median imputation.
- **Categorical branch**: NLCD Land Cover infinite-to-NaN conversion, missing category imputation (-9999), and One-Hot Encoding.
- **Classifier**: `XGBClassifier` with tuned hyperparameters (`n_estimators=400`, `learning_rate=0.03`, `max_depth=3`).


In [ ]:
def make_spatial_pipeline():
    """
    Create a new preprocessing and XGBoost pipeline.

    Numeric predictors:
    infinity -> NaN -> training-fold median

    Land cover:
    infinity -> NaN -> explicit missing code -> one-hot encoding
    """

    numeric_pipeline = Pipeline(
        steps=[
            (
                "nonfinite_to_nan",
                FunctionTransformer(
                    np.nan_to_num,
                    kw_args={
                        "copy": True,
                        "nan": np.nan,
                        "posinf": np.nan,
                        "neginf": np.nan,
                    },
                    feature_names_out="one-to-one",
                )
            ),
            (
                "median_imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
        ]
    )

    landcover_pipeline = Pipeline(
        steps=[
            (
                "nonfinite_to_nan",
                FunctionTransformer(
                    np.nan_to_num,
                    kw_args={
                        "copy": True,
                        "nan": np.nan,
                        "posinf": np.nan,
                        "neginf": np.nan,
                    },
                    feature_names_out="one-to-one",
                )
            ),
            (
                "missing_category",
                SimpleImputer(
                    strategy="constant",
                    fill_value=MISSING_LANDCOVER_CODE
                )
            ),
            (
                "one_hot_encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            ),
            (
                "landcover",
                landcover_pipeline,
                categorical_features
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    classifier = XGBClassifier(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=2,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=SEED,
        n_jobs=4,
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                classifier
            ),
        ]
    )


### 7. Create Spatial Groups (Spatial Blocking)

Apply K-Means clustering on the spatial coordinates (`POINT_X`, `POINT_Y`) to partition points into 10 spatially contiguous groups. This prevents spatial autocorrelation leakage between training and testing folds.


In [ ]:
warnings.filterwarnings(
    "ignore",
    message=(
        "KMeans is known to have a memory leak "
        "on Windows with MKL.*"
    )
)

spatial_groups = KMeans(
    n_clusters=NUMBER_OF_SPATIAL_GROUPS,
    n_init=20,
    random_state=SEED,
).fit_predict(
    coordinates.to_numpy()
)

spatial_assignments = df[
    [
        "POINT_X",
        "POINT_Y",
        target_field,
    ]
].copy()

spatial_assignments[
    "spatial_group"
] = spatial_groups

spatial_assignments.to_csv(
    output_dir
    / "xgboost_spatial_group_assignments.csv",
    index=False
)

spatial_group_balance = pd.crosstab(
    pd.Series(
        spatial_groups,
        name="spatial_group"
    ),
    pd.Series(
        y.to_numpy(),
        name="label"
    ),
    margins=True
)

spatial_group_balance.to_csv(
    output_dir
    / "xgboost_spatial_group_balance.csv"
)

print("\nSpatial-group balance:")
display(spatial_group_balance)


### 8. Five-Fold Spatial Cross-Validation

Evaluate generalization using `GroupKFold` across spatial clusters. Compute fold-level and aggregated mean/std scores for Accuracy, Precision, Recall, F1, and ROC-AUC.


In [ ]:
spatial_cv = GroupKFold(
    n_splits=NUMBER_OF_CV_FOLDS
)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(
        precision_score,
        zero_division=0
    ),
    "recall": make_scorer(
        recall_score,
        zero_division=0
    ),
    "f1": make_scorer(
        f1_score,
        zero_division=0
    ),
    "roc_auc": "roc_auc",
}

cv_result = cross_validate(
    make_spatial_pipeline(),
    X_raw,
    y,
    groups=spatial_groups,
    cv=spatial_cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=False,
)

fold_details = []

for fold_number, (
    fold_train_indices,
    fold_test_indices
) in enumerate(
    spatial_cv.split(
        X_raw,
        y,
        groups=spatial_groups
    ),
    start=1
):
    fold_test_groups = sorted(
        np.unique(
            spatial_groups[
                fold_test_indices
            ]
        ).astype(int).tolist()
    )

    fold_test_labels = y.iloc[
        fold_test_indices
    ]

    fold_details.append({
        "fold": fold_number,
        "n_train":
            len(fold_train_indices),
        "n_test":
            len(fold_test_indices),
        "test_groups":
            ",".join(
                map(str, fold_test_groups)
            ),
        "test_non_landslide":
            int(
                (fold_test_labels == 0).sum()
            ),
        "test_landslide":
            int(
                (fold_test_labels == 1).sum()
            ),
    })

fold_details = pd.DataFrame(
    fold_details
)

cv_table = pd.DataFrame({
    "fold": range(
        1,
        NUMBER_OF_CV_FOLDS + 1
    ),
    "accuracy":
        cv_result["test_accuracy"],
    "precision":
        cv_result["test_precision"],
    "recall":
        cv_result["test_recall"],
    "f1":
        cv_result["test_f1"],
    "roc_auc":
        cv_result["test_roc_auc"],
})

cv_table = cv_table.merge(
    fold_details,
    on="fold",
    how="left"
)

metric_columns = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
]

cv_mean = cv_table[
    metric_columns
].mean()

cv_standard_deviation = cv_table[
    metric_columns
].std(ddof=1)

cv_summary = pd.concat(
    [
        cv_table,
        pd.DataFrame([
            {
                "fold": "mean",
                **cv_mean.to_dict(),
            },
            {
                "fold": "std",
                **cv_standard_deviation.to_dict(),
            },
        ]),
    ],
    ignore_index=True
)

cv_summary.to_csv(
    output_dir
    / "xgboost_spatial_cross_validation_metrics.csv",
    index=False
)

print("\nSpatial cross-validation:")
display(cv_summary)


### 9. Spatial Out-Of-Fold (OOF) Predictions

Generate unbiased out-of-fold susceptibility probabilities for every inventory point using models trained strictly without that spatial block.


In [ ]:
oof_probability = cross_val_predict(
    make_spatial_pipeline(),
    X_raw,
    y,
    groups=spatial_groups,
    cv=spatial_cv,
    method="predict_proba",
    n_jobs=1,
)[:, 1]

oof_prediction = (
    oof_probability
    >= CLASSIFICATION_THRESHOLD
).astype(int)

oof_metrics = pd.DataFrame([{
    "accuracy": accuracy_score(
        y,
        oof_prediction
    ),
    "precision": precision_score(
        y,
        oof_prediction,
        zero_division=0
    ),
    "recall": recall_score(
        y,
        oof_prediction,
        zero_division=0
    ),
    "f1": f1_score(
        y,
        oof_prediction,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y,
        oof_probability
    ),
    "prediction_type":
        "SPATIAL_OUT_OF_FOLD_VALIDATION",
}])

oof_metrics.to_csv(
    output_dir
    / "xgboost_spatial_oof_metrics.csv",
    index=False
)

oof_predictions = df.copy()

oof_predictions[
    "spatial_group"
] = spatial_groups

oof_predictions[
    "xgb_oof_susceptibility_score"
] = oof_probability

oof_predictions[
    "xgb_oof_predicted_class"
] = oof_prediction

oof_predictions[
    "prediction_type"
] = "SPATIAL_OUT_OF_FOLD_VALIDATION"

oof_predictions.to_csv(
    output_dir
    / "xgboost_spatial_oof_predictions.csv",
    index=False
)

print("\nSpatial out-of-fold metrics:")
display(oof_metrics)


### 10. Cross-Validation Performance Graph

Plot mean performance metrics across spatial cross-validation with error bars indicating fold standard deviation.


In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 6)
)

bars = ax.bar(
    [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC",
    ],
    cv_mean[metric_columns],
    yerr=cv_standard_deviation[
        metric_columns
    ],
    capsize=5,
    color="#326273"
)

ax.set_ylim(0, 1.05)
ax.set_ylabel("Spatial Cross-Validation Score")

ax.set_title(
    "Five-Fold Spatial Cross-Validation Performance",
    fontweight="bold"
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.30
)

ax.set_axisbelow(True)

ax.bar_label(
    bars,
    labels=[
        f"{value:.3f}"
        for value in
        cv_mean[metric_columns]
    ],
    padding=5
)

fig.tight_layout()

fig.savefig(
    output_dir
    / "xgboost_spatial_cross_validation.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)


### 11. Spatially Separated Holdout Split

Withhold 30% of spatial clusters as a dedicated spatial holdout test set using `GroupShuffleSplit` for independent model verification.


In [ ]:
spatial_holdout_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=SPATIAL_HOLDOUT_GROUP_FRACTION,
    random_state=SEED,
)

holdout_train_indices, holdout_test_indices = next(
    spatial_holdout_splitter.split(
        X_raw,
        y,
        groups=spatial_groups
    )
)

holdout_train_groups = sorted(
    np.unique(
        spatial_groups[
            holdout_train_indices
        ]
    ).astype(int).tolist()
)

holdout_test_groups = sorted(
    np.unique(
        spatial_groups[
            holdout_test_indices
        ]
    ).astype(int).tolist()
)

X_holdout_train = X_raw.iloc[
    holdout_train_indices
]

X_holdout_test = X_raw.iloc[
    holdout_test_indices
]

y_holdout_train = y.iloc[
    holdout_train_indices
]

y_holdout_test = y.iloc[
    holdout_test_indices
]

if y_holdout_train.nunique() < 2:
    raise ValueError(
        "Spatial holdout training data contains only one class."
    )

if y_holdout_test.nunique() < 2:
    raise ValueError(
        "Spatial holdout testing data contains only one class."
    )

spatial_holdout_model = (
    make_spatial_pipeline()
)

spatial_holdout_model.fit(
    X_holdout_train,
    y_holdout_train
)

holdout_probability = (
    spatial_holdout_model
    .predict_proba(
        X_holdout_test
    )[:, 1]
)

holdout_prediction = (
    holdout_probability
    >= CLASSIFICATION_THRESHOLD
).astype(int)

holdout_accuracy = accuracy_score(
    y_holdout_test,
    holdout_prediction
)

holdout_precision = precision_score(
    y_holdout_test,
    holdout_prediction,
    zero_division=0
)

holdout_recall = recall_score(
    y_holdout_test,
    holdout_prediction,
    zero_division=0
)

holdout_f1 = f1_score(
    y_holdout_test,
    holdout_prediction,
    zero_division=0
)

holdout_auc = roc_auc_score(
    y_holdout_test,
    holdout_probability
)

spatial_holdout_metrics = pd.DataFrame([{
    "accuracy":
        holdout_accuracy,
    "precision":
        holdout_precision,
    "recall":
        holdout_recall,
    "f1":
        holdout_f1,
    "roc_auc":
        holdout_auc,
    "classification_threshold":
        CLASSIFICATION_THRESHOLD,
    "n_train":
        len(holdout_train_indices),
    "n_test":
        len(holdout_test_indices),
    "training_groups":
        ",".join(
            map(str, holdout_train_groups)
        ),
    "testing_groups":
        ",".join(
            map(str, holdout_test_groups)
        ),
    "test_non_landslide":
        int((y_holdout_test == 0).sum()),
    "test_landslide":
        int((y_holdout_test == 1).sum()),
    "prediction_type":
        "SPATIAL_GROUP_HOLDOUT",
}])

spatial_holdout_metrics.to_csv(
    output_dir
    / "xgboost_spatial_holdout_metrics.csv",
    index=False
)

print("\nSpatial holdout metrics:")
display(spatial_holdout_metrics)


### 12. Spatial Holdout Predictions

Predict susceptibility scores on the spatially isolated holdout set and flag correct/incorrect classifications.


In [ ]:
spatial_holdout_predictions = df.iloc[
    holdout_test_indices
].copy()

spatial_holdout_predictions[
    "spatial_group"
] = spatial_groups[
    holdout_test_indices
]

spatial_holdout_predictions[
    "xgb_susceptibility_score"
] = holdout_probability

spatial_holdout_predictions[
    "xgb_predicted_class"
] = holdout_prediction

spatial_holdout_predictions[
    "prediction_correct"
] = (
    y_holdout_test.to_numpy()
    == holdout_prediction
)

spatial_holdout_predictions[
    "prediction_type"
] = "SPATIAL_GROUP_HOLDOUT"

spatial_holdout_predictions.to_csv(
    output_dir
    / "xgboost_spatial_holdout_predictions.csv",
    index=False
)


### 13. Classification Report

Generate and export detailed precision, recall, and F1-scores per class for the spatial holdout.


In [ ]:
holdout_report = classification_report(
    y_holdout_test,
    holdout_prediction,
    labels=[0, 1],
    target_names=[
        "Non-Landslide",
        "Landslide",
    ],
    output_dict=True,
    zero_division=0,
)

holdout_report_table = (
    pd.DataFrame(
        holdout_report
    ).transpose()
)

holdout_report_table.to_csv(
    output_dir
    / "xgboost_spatial_holdout_classification_report.csv"
)

print("\nSpatial holdout classification report:")
display(holdout_report_table)


### 14. Explained Confusion Matrix

Construct a publication-ready confusion matrix with True Negatives, False Positives, False Negatives, and True Positives, accompanied by an explanatory callout box detailing Precision, Recall, Specificity, and F1-Score.


In [ ]:
holdout_confusion = confusion_matrix(
    y_holdout_test,
    holdout_prediction,
    labels=[0, 1]
)

# sklearn order:
# [[True Negative, False Positive],
#  [False Negative, True Positive]]

tn, fp, fn, tp = (
    holdout_confusion.ravel()
)

holdout_specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else np.nan
)

holdout_confusion_table = pd.DataFrame(
    holdout_confusion,
    index=[
        "Observed_Non-Landslide",
        "Observed_Landslide",
    ],
    columns=[
        "Predicted_Non-Landslide",
        "Predicted_Landslide",
    ]
)

holdout_confusion_table.to_csv(
    output_dir
    / "xgboost_spatial_holdout_confusion_matrix.csv"
)

explained_metrics = pd.DataFrame([{
    "true_negative_TN":
        int(tn),
    "false_positive_FP":
        int(fp),
    "false_negative_FN":
        int(fn),
    "true_positive_TP":
        int(tp),
    "accuracy":
        holdout_accuracy,
    "precision":
        holdout_precision,
    "recall_sensitivity":
        holdout_recall,
    "specificity":
        holdout_specificity,
    "f1_score":
        holdout_f1,
    "roc_auc":
        holdout_auc,
}])

explained_metrics.to_csv(
    output_dir
    / "xgboost_spatial_holdout_explained_metrics.csv",
    index=False
)

fig, ax = plt.subplots(
    figsize=(10, 7)
)

sns.heatmap(
    holdout_confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    square=True,
    linewidths=0.5,
    linecolor="#D0D0D0",
    annot_kws={
        "fontsize": 16
    },
    xticklabels=[
        "Non-Landslide",
        "Landslide",
    ],
    yticklabels=[
        "Non-Landslide",
        "Landslide",
    ],
    ax=ax
)

ax.set_title(
    "XGBoost Spatial Holdout\n"
    f"AUC={holdout_auc:.4f} | "
    f"Acc={holdout_accuracy:.4f}",
    fontsize=16,
    fontweight="bold",
    pad=12
)

ax.set_xlabel(
    "Predicted Label",
    fontsize=12
)

ax.set_ylabel(
    "True Label",
    fontsize=12
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=0
)

ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0
)

metrics_text = (
    f"TP = {tp}\n"
    f"TN = {tn}\n"
    f"FP = {fp}\n"
    f"FN = {fn}\n"
    "\n"
    f"Precision = {holdout_precision:.3f}\n"
    f"Recall = {holdout_recall:.3f}\n"
    f"Specificity = {holdout_specificity:.3f}\n"
    f"F1-Score = {holdout_f1:.3f}"
)

ax.text(
    1.08,
    0.50,
    metrics_text,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="center",
    horizontalalignment="left",
    bbox={
        "boxstyle": "round,pad=0.4",
        "facecolor": "#FFFDE7",
        "edgecolor": "#333333",
        "linewidth": 1.0,
    }
)

fig.subplots_adjust(
    left=0.13,
    right=0.73,
    bottom=0.13,
    top=0.82
)

fig.savefig(
    output_dir
    / "xgboost_spatial_holdout_confusion_matrix_explained.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

print("\nExplained holdout metrics:")
display(explained_metrics)


### 15. Spatial Holdout ROC Curve

Plot the Receiver Operating Characteristic (ROC) curve and calculate Area Under the Curve (AUC) for the spatial holdout set.


In [ ]:
false_positive_rate, true_positive_rate, roc_thresholds = (
    roc_curve(
        y_holdout_test,
        holdout_probability
    )
)

roc_table = pd.DataFrame({
    "false_positive_rate":
        false_positive_rate,
    "true_positive_rate":
        true_positive_rate,
    "threshold":
        roc_thresholds,
})

roc_table.to_csv(
    output_dir
    / "xgboost_spatial_holdout_roc_curve.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(6, 5))

ax.plot(
    false_positive_rate,
    true_positive_rate,
    linewidth=2,
    color="#C44E52",
    label=(
        f"Spatial holdout AUC = "
        f"{holdout_auc:.3f}"
    )
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    label="Random classifier"
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")

ax.set_title(
    "XGBoost Spatial Holdout ROC Curve",
    fontweight="bold"
)

ax.legend(loc="lower right")

ax.grid(
    linestyle="--",
    alpha=0.25
)

fig.tight_layout()

fig.savefig(
    output_dir
    / "xgboost_spatial_holdout_roc_curve.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)


### 16. Save Spatial Holdout Pipeline

Serialize the trained spatial holdout pipeline to disk as a `.joblib` artifact.


In [ ]:
spatial_holdout_pipeline_path = (
    output_dir
    / "xgboost_kwri_spatial_holdout_pipeline.joblib"
)

joblib.dump(
    spatial_holdout_model,
    spatial_holdout_pipeline_path
)


### 17. Train Final Deployment Pipeline

Train the complete pipeline on all available inventory records to prepare the final model for regional susceptibility map generation.


In [ ]:
# Used for susceptibility mapping only.
# It is trained on all available points.

final_pipeline = (
    make_spatial_pipeline()
)

final_pipeline.fit(
    X_raw,
    y
)

final_pipeline_path = (
    output_dir
    / "xgboost_kwri_spatial_final_pipeline.joblib"
)

joblib.dump(
    final_pipeline,
    final_pipeline_path
)


### 18. In-Sample Fitted Predictions

Compute fitted in-sample predictions for full dataset inspection, explicitly tagging records as `IN_SAMPLE_NOT_VALIDATION`.


In [ ]:
in_sample_probability = (
    final_pipeline
    .predict_proba(X_raw)[:, 1]
)

in_sample_prediction = (
    in_sample_probability
    >= CLASSIFICATION_THRESHOLD
).astype(int)

in_sample_predictions = df.copy()

in_sample_predictions[
    "xgb_in_sample_susceptibility_score"
] = in_sample_probability

in_sample_predictions[
    "xgb_in_sample_predicted_class"
] = in_sample_prediction

in_sample_predictions[
    "prediction_type"
] = "IN_SAMPLE_NOT_VALIDATION"

in_sample_predictions.to_csv(
    output_dir
    / "xgboost_all_point_predictions_IN_SAMPLE.csv",
    index=False
)


### 19. Feature Display Names (NLCD Mapping)

Define translation mappings for NLCD Land Cover numeric codes (e.g. 41 -> Deciduous Forest, 81 -> Pasture/Hay) and clean feature labels for clear graphical reporting.


In [ ]:
nlcd_names = {
    MISSING_LANDCOVER_CODE:
        "Missing/Unknown",
    0:
        "NoData / Unclassified",
    11:
        "Open Water",
    12:
        "Perennial Ice/Snow",
    21:
        "Developed, Open Space",
    22:
        "Developed, Low Intensity",
    23:
        "Developed, Medium Intensity",
    24:
        "Developed, High Intensity",
    31:
        "Barren Land",
    41:
        "Deciduous Forest",
    42:
        "Evergreen Forest",
    43:
        "Mixed Forest",
    51:
        "Dwarf Scrub",
    52:
        "Shrub/Scrub",
    71:
        "Grassland/Herbaceous",
    72:
        "Sedge/Herbaceous",
    73:
        "Lichens",
    74:
        "Moss",
    81:
        "Pasture/Hay",
    82:
        "Cultivated Crops",
    90:
        "Woody Wetlands",
    95:
        "Emergent Herbaceous Wetlands",
}

regular_names = {
    "Curvature":
        "Curvature",
    "TWI":
        "Topographic Wetness Index (TWI)",
    "Dist_Road_KYTC_1":
        "Distance to KYTC Roads",
    "Plan_Curv":
        "Plan Curvature",
    "Profile_Curv":
        "Profile Curvature",
    "Distan_to_Rivers":
        "Distance to Rivers/Streams",
    "Slope_KWRI":
        "Slope",
    "Aspect_KWRI":
        "Aspect",
    "Rain_Mean_KWRI":
        "Mean Rainfall",
    "NDVI_Mean_KWRI":
        "NDVI",
}


def readable_feature_name(feature):
    feature = str(feature)
    prefix = "Landcover_KWRI_"

    if feature.startswith(prefix):
        raw_code = feature[
            len(prefix):
        ]

        try:
            code = int(float(raw_code))

            class_name = nlcd_names.get(
                code,
                "Unknown NLCD Class"
            )

            return (
                f"Land Cover: {code} "
                f"({class_name})"
            )

        except (ValueError, TypeError):
            return (
                "Land Cover: Missing/Unknown"
            )

    return regular_names.get(
        feature,
        feature
    )


def safe_reporting_name(name):
    cleaned = re.sub(
        r"[^0-9A-Za-z_]+",
        "_",
        str(name)
    )

    cleaned = re.sub(
        r"_+",
        "_",
        cleaned
    ).strip("_")

    return cleaned or "feature"


### 20. Detailed Feature Importance

Extract and rank predictor importances from the fitted XGBoost booster, including individual one-hot encoded land cover categories.


In [ ]:
fitted_preprocessor = (
    final_pipeline
    .named_steps["preprocessor"]
)

fitted_classifier = (
    final_pipeline
    .named_steps["classifier"]
)

encoded_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)

importance = pd.DataFrame({
    "feature":
        encoded_feature_names,
    "importance":
        fitted_classifier.feature_importances_,
})

importance["safe_feature_name"] = (
    importance["feature"]
    .map(safe_reporting_name)
)

importance["is_landcover"] = (
    importance["feature"]
    .str.startswith("Landcover_KWRI_")
)

importance["display_feature"] = (
    importance["feature"]
    .map(readable_feature_name)
)

importance["importance_percent"] = (
    importance["importance"] * 100
)

importance = (
    importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

importance.insert(
    0,
    "rank",
    np.arange(
        1,
        len(importance) + 1
    )
)

importance.to_csv(
    output_dir
    / "xgboost_final_feature_importance_detailed.csv",
    index=False
)

print("\nDetailed feature importance:")
display(importance)


### 21. Grouped Feature Importance (Land Cover Total)

Aggregate all one-hot encoded land cover categories into a single unified `Land Cover Total` metric to evaluate overall land cover contribution.


In [ ]:
importance["grouped_feature"] = np.where(
    importance["is_landcover"],
    "Land Cover Total",
    importance["display_feature"]
)

grouped_importance = (
    importance
    .groupby(
        "grouped_feature",
        as_index=False
    )["importance"]
    .sum()
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

grouped_importance[
    "importance_percent"
] = (
    grouped_importance["importance"]
    * 100
)

grouped_importance.insert(
    0,
    "rank",
    np.arange(
        1,
        len(grouped_importance) + 1
    )
)

grouped_importance.to_csv(
    output_dir
    / "xgboost_final_feature_importance_grouped.csv",
    index=False
)

print("\nGrouped feature importance:")
display(grouped_importance)


### 22. Shared Importance Scale

Calculate a consistent x-axis scale across both detailed and grouped importance charts for direct visual comparison.


In [ ]:
maximum_importance = max(
    importance["importance"].max(),
    grouped_importance["importance"].max()
)

common_axis_max = np.ceil(
    maximum_importance
    * 1.10
    * 10
) / 10

if common_axis_max <= 0:
    common_axis_max = 1.0


### 23. Detailed Feature-Importance Graph

Render a horizontal bar plot showing each predictor's relative importance percentage.


In [ ]:
detailed_plot = (
    importance[
        importance["importance"] > 0
    ]
    .sort_values(
        "importance",
        ascending=True
    )
    .copy()
)

detailed_colors = np.where(
    detailed_plot["is_landcover"],
    "#5A8F4F",
    "#326273"
)

figure_height = max(
    7,
    1.5 + len(detailed_plot) * 0.43
)

fig, ax = plt.subplots(
    figsize=(12, figure_height)
)

bars = ax.barh(
    detailed_plot["display_feature"],
    detailed_plot["importance"],
    color=detailed_colors
)

ax.set_xlim(0, common_axis_max)
ax.set_xlabel("XGBoost Feature Importance")
ax.set_ylabel("")

ax.set_title(
    "Final XGBoost Predictor Importance",
    fontweight="bold"
)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.30
)

ax.set_axisbelow(True)

ax.bar_label(
    bars,
    labels=[
        f"{value:.3f}"
        for value in
        detailed_plot["importance"]
    ],
    padding=4
)

fig.tight_layout()

fig.savefig(
    output_dir
    / "xgboost_final_feature_importance_detailed.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)


### 24. Grouped Feature-Importance Graph

Render the grouped horizontal bar chart highlighting the aggregated contribution of Land Cover versus topographic predictors.


In [ ]:
grouped_plot = (
    grouped_importance[
        grouped_importance["importance"] > 0
    ]
    .sort_values(
        "importance",
        ascending=True
    )
    .copy()
)

grouped_colors = [
    "#C45A3D"
    if name == "Land Cover Total"
    else "#326273"
    for name in
    grouped_plot["grouped_feature"]
]

fig, ax = plt.subplots(
    figsize=(12, 7)
)

bars = ax.barh(
    grouped_plot["grouped_feature"],
    grouped_plot["importance"],
    color=grouped_colors
)

ax.set_xlim(0, common_axis_max)
ax.set_xlabel("XGBoost Feature Importance")
ax.set_ylabel("")

ax.set_title(
    "Grouped XGBoost Predictor Importance",
    fontweight="bold"
)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.30
)

ax.set_axisbelow(True)

ax.bar_label(
    bars,
    labels=[
        f"{value:.3f}"
        for value in
        grouped_plot["importance"]
    ],
    padding=4
)

fig.tight_layout()

fig.savefig(
    output_dir
    / "xgboost_final_feature_importance_grouped.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)


### 25. Save Final Model Bundle

Export a unified model bundle dictionary containing the fitted pipeline, feature lists, thresholds, and configuration metadata.


In [ ]:
model_bundle = {
    "pipeline":
        final_pipeline,
    "raw_features":
        raw_features,
    "numeric_features":
        numeric_features,
    "categorical_features":
        categorical_features,
    "missing_landcover_code":
        MISSING_LANDCOVER_CODE,
    "classification_threshold":
        CLASSIFICATION_THRESHOLD,
    "scale_pos_weight":
        scale_pos_weight,
    "number_of_spatial_groups":
        NUMBER_OF_SPATIAL_GROUPS,
    "number_of_cv_folds":
        NUMBER_OF_CV_FOLDS,
    "random_seed":
        SEED,
}

model_bundle_path = (
    output_dir
    / "xgboost_kwri_spatial_model_bundle.joblib"
)

joblib.dump(
    model_bundle,
    model_bundle_path
)


### 26. Save Run Settings & Metadata

Export complete experiment configuration, hyperparameter settings, sample sizes, and file paths as a structured JSON file.


In [ ]:
run_settings = {
    "input_csv":
        str(input_csv),
    "output_folder":
        str(output_dir),
    "number_of_records":
        int(len(df)),
    "class_counts": {
        "non_landslide":
            negative_count,
        "landslide":
            positive_count,
    },
    "scale_pos_weight":
        float(scale_pos_weight),
    "raw_features":
        raw_features,
    "number_of_spatial_groups":
        NUMBER_OF_SPATIAL_GROUPS,
    "number_of_cv_folds":
        NUMBER_OF_CV_FOLDS,
    "spatial_holdout_group_fraction":
        SPATIAL_HOLDOUT_GROUP_FRACTION,
    "spatial_holdout_training_groups":
        holdout_train_groups,
    "spatial_holdout_testing_groups":
        holdout_test_groups,
    "classification_threshold":
        CLASSIFICATION_THRESHOLD,
    "random_seed":
        SEED,
    "primary_evaluation":
        "five_fold_spatial_cross_validation",
    "secondary_evaluation":
        "spatial_group_holdout",
    "mapping_pipeline":
        str(final_pipeline_path),
    "model_bundle":
        str(model_bundle_path),
    "all_point_prediction_warning":
        "IN_SAMPLE_NOT_VALIDATION",
}

with (
    output_dir
    / "xgboost_spatial_run_settings.json"
).open(
    "w",
    encoding="utf-8"
) as settings_file:
    json.dump(
        run_settings,
        settings_file,
        indent=2
    )


### 27. Output Inventory Audit

Inspect the output directory and catalog all generated CSVs, figures, models, and metadata tables with byte sizes.


In [ ]:
result_files = sorted(
    [
        path
        for path in output_dir.iterdir()
        if path.is_file()
    ],
    key=lambda path: path.name.lower()
)

inventory = pd.DataFrame({
    "file_name": [
        path.name
        for path in result_files
    ],
    "file_size_bytes": [
        path.stat().st_size
        for path in result_files
    ],
    "full_path": [
        str(path)
        for path in result_files
    ],
})

inventory.to_csv(
    output_dir
    / "xgboost_spatial_results_inventory.csv",
    index=False
)


### 28. Final Summary & Key Metrics

Display the summary dashboard of spatial cross-validation, holdout performance, and artifact locations.


In [ ]:
print("\n" + "=" * 70)
print("FINAL KWRI SPATIAL XGBOOST ANALYSIS COMPLETED")
print("=" * 70)

print("\nPrimary evaluation:")
print("Five-fold spatial cross-validation")

print("\nCross-validation mean:")
display(
    pd.DataFrame(
        cv_mean,
        columns=["mean_score"]
    )
)

print("\nSpatial out-of-fold metrics:")
display(oof_metrics)

print("\nSecondary spatial holdout metrics:")
display(spatial_holdout_metrics)

print("\nExplained confusion-matrix metrics:")
display(explained_metrics)

print("\nFinal mapping pipeline:")
print(final_pipeline_path)

print("\nFinal model bundle:")
print(model_bundle_path)

print("\nResults folder:")
print(output_dir)

print(
    "\nAll-point fitted predictions are labeled "
    "IN_SAMPLE_NOT_VALIDATION."
)

print("\nSaved files:")
display(inventory)
